# Notebook 09: Hyperparameter Tuning

## Why This Matters

Hyperparameter tuning is where theoretical model selection becomes practical model performance. Bergstra & Bengio (2012) demonstrated that random search outperforms grid search by exploiting the low effective dimensionality of hyperparameter spaces. Bayesian optimization (Optuna, Hyperopt) pushes this further by learning a surrogate model of the objective, directing search toward promising regions. In industry, the difference between naive and smart tuning routinely accounts for 2-5% AUC improvement -- enough to matter in competitive ML applications.

## Background

**Grid search** exhaustively evaluates every combination -- it scales exponentially with the number of hyperparameters. **Random search** samples uniformly from distributions, requiring far fewer trials because most parameters have low interaction. **Bayesian optimization** fits a Gaussian process or TPE (Tree-structured Parzen Estimator) surrogate, then uses an acquisition function (EI, UCB) to decide where to evaluate next.

**Key references:**
- Bergstra & Bengio 2012: "Random Search for Hyper-Parameter Optimization"
- Akiba et al. 2019: "Optuna: A Next-generation Hyperparameter Optimization Framework"
- Hutter et al. 2011: SMAC (Sequential Model-Based Algorithm Configuration)

## Community Context

Optuna has become the de-facto standard in Kaggle and production ML (used by Preferred Networks, NTT, and many others). Its define-by-run API lets you prune unpromising trials mid-training -- critical when training time is expensive. HalvingGridSearchCV (sklearn 0.24+) applies successive halving to eliminate poor candidates early.

In [ ]:
%pip install -q optuna scikit-learn xgboost numpy pandas matplotlib lightgbm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, cross_val_score,
    StratifiedKFold, KFold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingGridSearchCV
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)

# Synthetic classification dataset
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, n_clusters_per_class=2, random_state=42
)
print(f"Dataset: {X.shape}, positive rate: {y.mean():.2f}")

## 1. Grid Search vs Random Search

Grid search is guaranteed to find the best combination in the grid, but the grid must be manually designed and grows exponentially. Random search samples the same budget more efficiently because hyperparameters rarely interact equally -- a few parameters dominate performance, and random search automatically allocates more exploration to important dimensions.

In [ ]:
from scipy.stats import randint, uniform

# Grid search
param_grid = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_depth": [3, 5, 10, None],
    "rf__min_samples_split": [2, 5, 10]
}
pipe = Pipeline([
    ("rf", RandomForestClassifier(random_state=42))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid_search.fit(X, y)
print(f"Grid search: {len(grid_search.cv_results_['mean_test_score'])} fits")
print(f"Grid best AUC: {grid_search.best_score_:.4f}")
print(f"Grid best params: {grid_search.best_params_}")

# Random search
param_dist = {
    "rf__n_estimators": randint(50, 300),
    "rf__max_depth": [3, 5, 7, 10, None],
    "rf__min_samples_split": randint(2, 20),
    "rf__max_features": uniform(0.3, 0.7)
}
rand_search = RandomizedSearchCV(
    pipe, param_dist, n_iter=36, cv=cv, scoring="roc_auc",
    n_jobs=-1, random_state=42
)
rand_search.fit(X, y)
print(f"
Random search: {len(rand_search.cv_results_['mean_test_score'])} fits")
print(f"Random best AUC: {rand_search.best_score_:.4f}")
print(f"Random best params: {rand_search.best_params_}")

## 2. Bayesian Optimization with Optuna

Optuna uses TPE (Tree-structured Parzen Estimator) to model the distribution of hyperparameters in good trials vs. bad trials. It selects new candidates by maximizing the expected improvement -- effectively learning which regions of hyperparameter space to explore. The define-by-run API means the search space can be conditional (e.g., only tune `colsample_bytree` if `booster="gbtree"`).

In [ ]:
# Optuna objective for XGBoost
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "eval_metric": "logloss",
    }
    model = xgb.XGBClassifier(**params, verbosity=0)
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(xgb_objective, n_trials=50, show_progress_bar=False)

print(f"Optuna best AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

# Convergence plot
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trial_values, "o", alpha=0.4, markersize=4, label="Trial AUC")
ax.plot(best_so_far, "r-", linewidth=2, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("AUC")
ax.set_title("Optuna Convergence (XGBoost)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Trials: {len(study.trials)}")

## 3. Pruning with Early Stopping

Optuna can prune unpromising trials mid-training via callbacks. For XGBoost, each boosting round reports an intermediate value, and the pruner terminates trials that are unlikely to beat the current best. This reduces compute by 3-5x on large datasets.

In [ ]:
# Optuna with pruning via sklearn early stopping workaround
# (XGBoost native early stopping + Optuna callback)
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def xgb_pruning_objective(trial):
    params = {
        "n_estimators": 500,
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "eval_metric": "auc",
        "early_stopping_rounds": 20,
        "verbosity": 0,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    auc = model.best_score
    trial.report(auc, step=model.best_iteration)
    if trial.should_prune():
        raise optuna.TrialPruned()
    return auc

pruning_study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    sampler=optuna.samplers.TPESampler(seed=42)
)
pruning_study.optimize(xgb_pruning_objective, n_trials=30, show_progress_bar=False)

pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in pruning_study.trials)
complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in pruning_study.trials)
print(f"Pruned: {pruned}, Complete: {complete}")
print(f"Best AUC: {pruning_study.best_value:.4f}")

## 4. HalvingGridSearchCV (Successive Halving)

Successive halving starts all candidates with a small resource budget (few training samples or estimators) and iteratively doubles resources while halving the number of candidates -- keeping only the top 50% each round. This finds near-optimal configurations with $O(\log N)$ total resources vs. $O(N)$ for full grid search.

In [ ]:
param_grid_rf = {
    "n_estimators": [50, 100, 150, 200, 250],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5]
}

rf = RandomForestClassifier(random_state=42)
halving_cv = HalvingGridSearchCV(
    rf, param_grid_rf,
    resource="n_samples",
    factor=2,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=0
)
halving_cv.fit(X, y)

results_df = pd.DataFrame(halving_cv.cv_results_)
print(f"Total fits: {len(results_df)}")
print(f"Iterations: {results_df['iter'].nunique()}")
print(f"Best AUC: {halving_cv.best_score_:.4f}")
print(f"Best params: {halving_cv.best_params_}")

# Show progression
for it in sorted(results_df["iter"].unique()):
    sub = results_df[results_df["iter"] == it]
    print(f"  Iter {it}: {len(sub)} candidates, "  
          f"n_resources={sub['n_resources'].iloc[0]}, "  
          f"best_AUC={sub['mean_test_score'].max():.4f}")

## 5. Nested Cross-Validation for Unbiased Evaluation

When you tune hyperparameters and evaluate on the same data, you get an optimistic estimate because the evaluation set "leaked" into the model selection process. Nested CV uses an outer loop for unbiased evaluation and an inner loop for hyperparameter selection. The outer test scores are unbiased estimates of generalization performance.

In [ ]:
from sklearn.model_selection import cross_validate

# Nested CV setup
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Model with inner CV for tuning
param_grid_small = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}
base_rf = RandomForestClassifier(random_state=42)
inner_search = GridSearchCV(base_rf, param_grid_small, cv=inner_cv, scoring="roc_auc", n_jobs=-1)

# Outer CV evaluation
nested_scores = cross_val_score(inner_search, X, y, cv=outer_cv, scoring="roc_auc", n_jobs=-1)

# Non-nested (optimistic) estimate
non_nested_search = GridSearchCV(base_rf, param_grid_small, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
non_nested_search.fit(X, y)
non_nested_score = non_nested_search.best_score_

print("Nested CV scores (unbiased):")
for i, s in enumerate(nested_scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"Nested CV mean +/- std: {nested_scores.mean():.4f} +/- {nested_scores.std():.4f}")
print(f"
Non-nested (optimistic) best score: {non_nested_score:.4f}")
print(f"Optimism bias: {non_nested_score - nested_scores.mean():.4f}")

## 6. Hyperparameter Importance (fANOVA)

Optuna can compute hyperparameter importance via fANOVA (functional ANOVA) -- it measures how much variance in the objective each hyperparameter explains. This guides future search by focusing on the parameters that matter most.

In [ ]:
import optuna

# Use the study from section 2
try:
    importance = optuna.importance.get_param_importances(study)
    print("Hyperparameter Importances (fANOVA):")
    for param, imp in sorted(importance.items(), key=lambda x: -x[1]):
        bar = "#" * int(imp * 40)
        print(f"  {param:30s} {imp:.4f} |{bar}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(8, 5))
    params = list(importance.keys())
    values = [importance[p] for p in params]
    idx = np.argsort(values)[::-1]
    ax.barh([params[i] for i in idx], [values[i] for i in idx], color="steelblue")
    ax.set_xlabel("Relative Importance")
    ax.set_title("Hyperparameter Importance (fANOVA)")
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Note: {e}")
    print("fANOVA requires scikit-learn 0.22+ and a completed study with >= 4 complete trials.")

## Summary

| Method | Search Budget | Key Advantage | Key Limitation |
|--------|--------------|---------------|----------------|
| Grid Search | O(n^k) | Exhaustive, reproducible | Exponential scaling |
| Random Search | Fixed n_iter | Efficient for low effective dim | No learning across trials |
| Bayesian (Optuna TPE) | Fixed n_trials | Learns from history, efficient | Overhead per trial |
| HalvingGridSearchCV | O(N log N) | Resource-efficient | Needs resource parameter |
| Nested CV | outer x inner | Unbiased generalization estimate | Computationally expensive |

**Interview tip:** Always distinguish between hyperparameter tuning (model selection) and evaluation (nested CV). Using the same validation fold for both inflates reported performance -- a common mistake in academic papers and Kaggle submissions alike.